In [1]:
import pandas as pd
import numpy as np
from pandas import Series, DataFrame

In [2]:
df = pd.read_csv(
    '../../../data/celebrity_deaths_2016.csv',
    usecols= ['dateofdeath', 'age']
)

In [3]:
df.head()

,dateofdeath,age
0,2016-01-01,71
1,2016-01-01,74
2,2016-01-01,79
3,2016-01-01,45
4,2016-01-01,83


In [4]:
df.dtypes

dateofdeath    str
age            str
dtype: object

In [5]:
# is there any NaN in the age or possible to convert int16/int8 format

df.isna().sum()

dateofdeath     0
age            27
dtype: int64

In [6]:
df['year'] = df['dateofdeath'].str.slice(0,4)
df['month'] = df['dateofdeath'].str.slice(5,7)
df['day'] = df['dateofdeath'].str.slice(-2) # NOTE: this counts from the back

df

,dateofdeath,age,year,month,day
0,2016-01-01,71,2016,01,01
1,2016-01-01,74,2016,01,01
2,2016-01-01,79,2016,01,01
3,2016-01-01,45,2016,01,01
4,2016-01-01,83,2016,01,01
...,...,...,...,...,...
6538,2016-12-27,74,2016,12,27
6539,2016-12-27,85,2016,12,27
6540,2016-12-27,83,2016,12,27
6541,2016-12-27,23,2016,12,27


In [7]:
df.age.max()

'99'

In [8]:
# Make the month column the index of the data frame.
df_2 = (
    df
    .copy()
    .set_index('month')
    .sort_index()
    ['age'].apply(lambda x: np.nan if not str(x).isdigit() else x).dropna().astype('int8')  
)


# There were some values in the age column that looked like numbers (e.g., 8889),
# but they were not valid digits because they contained whitespace or other non-visible characters.

In [9]:
df_2.info()

<class 'pandas.Series'>
Index: 6481 entries, 01 to 12
Series name: age
Non-Null Count  Dtype
--------------  -----
6481 non-null   int8 
dtypes: int8(1)
memory usage: 57.0+ KB


In [10]:
df_2

month
01    71
01    47
01    87
01    90
01    73
      ..
12    63
12    20
12    57
12    78
12    84
Name: age, Length: 6481, dtype: int8

In [11]:
# Find the average age of celebrities who died during that period (February–July 2016)

df_2.loc[slice('02','07')].mean()

np.float64(77.17887409200968)

#### Beyond the exercise_1
##### Add a new column, day, from the day of the month in which the celebrity died. Then create a multi-index (from `month` and `day`). What was the average age of death from Feb. 15 through July 15?

In [12]:
df_3 = (
    df
    .copy()
    .set_index(['month', 'day'])
    .sort_index()
)

In [13]:
df_3 = df_3['age'].apply(lambda x: np.nan if not str(x).isdigit() else x).dropna().astype('int8')

In [14]:
df_3

month  day
01     01     71
       01     74
       01     79
       01     45
       01     83
              ..
12     29     83
       29     88
       29     88
       29     89
       29     53
Name: age, Length: 6481, dtype: int8

In [15]:
df_3.loc[(slice('02', '07'), slice(None))].mean()

# NOTE: here I did not find the right solution

np.float64(77.17887409200968)

In [16]:
df_3.loc[('02', '15'):('07', '15')].mean()  
# NOTE: This is the correct solution from Reuven (it was a real "aha moment" when it finally clicked why this works)

np.float64(77.05183037332367)

#### Beyond the exercise_2
##### The CSV file contains another column, `causeofdeath`. Load it into a data frame, and find the five most common causes of death.

In [17]:
df['causeofdeath'] = pd.read_csv(
    '../../../data//celebrity_deaths_2016.csv',
    usecols= ['causeofdeath']
)

In [18]:
df.groupby('causeofdeath')[['causeofdeath']].value_counts().sort_values(ascending= False).reset_index().head(5)

,causeofdeath,count
0,cancer,248
1,heart attack,125
2,traffic collision,56
3,lung cancer,51
4,pneumonia,50


##### Now replace any NaN values in that column with the string `'unknown'`, and again find the five most common causes of death.

In [19]:
(
    df
    .fillna('unknown')
    .groupby('causeofdeath')[['causeofdeath']]
    .value_counts()
    .sort_values(ascending= False)
    .reset_index()
    .head(5)
)

,causeofdeath,count
0,unknown,5008
1,cancer,248
2,heart attack,125
3,traffic collision,56
4,lung cancer,51


#### Beyond the exercise_3
##### If someone asked whether cancer is in the top 10 causes, what would you say? Can you be more specific than that?

In [20]:
groupped_df = (
    df
    .fillna('unknown')
    .groupby('causeofdeath')[['causeofdeath']]
    .value_counts()
    .sort_values(ascending= False)
    .reset_index()
    .head(10)
)

In [21]:
cancer = (
    df
    .fillna('unknown')
    .groupby('causeofdeath')[['causeofdeath']]
    .value_counts()
    .sort_values(ascending= False)
    .reset_index()
)

sum_of_cancers = cancer[cancer['causeofdeath'].str.contains('cancer', case= False)]['count'].sum()

print(f'The {(sum_of_cancers/(groupped_df['count'].sum())*100).round(2)} % of  the deaths is caused by some kind of cancer')

The 9.28 % of  the deaths is caused by some kind of cancer
